# **CodeBLEU Benchmark**

In this notebook, we will evaluate the performance of a code generation model using the CodeBLEU metric. CodeBLEU is a widely used evaluation metric for code generation tasks that considers both the syntactic and semantic aspects of the generated code.

To compute the CodeBLEU score, we will use the `codebleu` library. We will compare the base model `Qwen2.5-1.5B-Instruct` with the fine-tuned model `Qwen2.5-1.5B-code-adapter` on a set of code generation tasks.

## CodeBLEU metrics

CodeBLEU combines four sub-metrics into a weighted score (here all weighted `0.25`). Each is in **[0, 1]** (higher = closer to the reference):

- **N-gram match** (`ngram_match_score`): classic BLEU — token n-gram overlap (lexical similarity).
- **Weighted n-gram match** (`weighted_ngram_match_score`): same, but gives more weight to language keywords (`if`, `for`, `return`, …).
- **Syntax match** (`syntax_match_score`): compares the abstract syntax trees (AST) — structural similarity.
- **Dataflow match** (`dataflow_match_score`): compares how variables are defined and used — semantic similarity.

The final `codebleu` is their weighted average, giving a fuller picture of code quality than BLEU alone.

Install the `codebleu` library.

In [48]:
%pip install -q --force-reinstall "tree-sitter==0.22.3" "tree-sitter-python==0.22.0"

ERROR: Ignored the following versions that require a different python version: 0.21.0 Requires-Python <3.12,>=3.8
ERROR: Could not find a version that satisfies the requirement tree-sitter-python==0.22.0 (from versions: 0.21.0, 0.23.0, 0.23.1, 0.23.2, 0.23.3, 0.23.4, 0.23.5, 0.23.6, 0.25.0)
ERROR: No matching distribution found for tree-sitter-python==0.22.0


In [49]:
%pip install -q torchao --upgrade

In [50]:
%pip install codebleu peft

Download libraries to test the matching of the generated code with the reference code.

In [51]:
%pip install -q --upgrade codebleu "tree-sitter>=0.23.0"
%pip install -q --upgrade \
    tree-sitter-python tree-sitter-ruby tree-sitter-java \
    tree-sitter-javascript tree-sitter-typescript \
    tree-sitter-go tree-sitter-php tree-sitter-c-sharp tree-sitter-c

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.2 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 4.3 MB/s eta 0:00:00


Import necessary libraries.

In [52]:
import json
from dataclasses import dataclass

In [53]:
import torch
from datasets import load_dataset
from codebleu import calc_codebleu
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

Create a dataclass to set the parameters for the CodeBLEU evaluation.

In [54]:
@dataclass
class EvalConfig:
    # Models
    original_model_name:  str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  
    finetuned_model_path: str = "./my_finetuned_qwen"                
 
    # Dataset
    dataset_name:  str = "Juanxxo/smallcoder-dataset"
    dataset_split: str = "test"      
    input_col:     str = "input_code"     
    output_col:    str = "target_completion"    
    max_samples:   int = 100         
 
     
    # CodeBLEU 
    lang:    str   = "python"
    weights: tuple = (0.25, 0.25, 0.25, 0.25)  
 
    # Generation 
    max_new_tokens: int = 256
    temperature:    float = 0.1      
    do_sample:      bool  = False    
    num_beams:      int   = 1        
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
 

Function to load the test dataset from Hugging Face Hub.

In [55]:
def load_test_samples(config: EvalConfig) -> list[dict]:
    print(f"Loading dataset: {config.dataset_name} (split='{config.dataset_split}')")
 
    try:
        ds = load_dataset(config.dataset_name, split=config.dataset_split)
    except ValueError as e:
        print(f" Split '{config.dataset_split}' not found: {e}")
        info = load_dataset(config.dataset_name)
        fallback = list(info.keys())[0]
        print(f"   → Using split '{fallback}' as fallback")
        ds = info[fallback]
 
    if config.max_samples and config.max_samples < len(ds):
        ds = ds.select(range(config.max_samples))
        print(f"Using {config.max_samples} of {len(ds)} samples")
    else:
        print(f"{len(ds)} samples loaded")
 
    for col in (config.input_col, config.output_col):
        if col not in ds.column_names:
            raise KeyError(
                f"Column '{col}' not found. "
                f"Available columns: {ds.column_names}"
            )
 
    return [
        {"input": row[config.input_col], "reference": row[config.output_col]}
        for row in ds
    ]

Load the models.

In [56]:
BASE_MODEL    = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_MODEL = "Juanxxo/qwen2.5-1.5B-code-adapter"

# ── Load the tokenizer (shared by both models)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Load base model only — adapter is loaded AFTER evaluating base
# (PeftModel.from_pretrained wraps the model in-place; loading it here
# would make both variables point to the same adapted weights)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()
print("Base model loaded successfully.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded successfully.


## **INFERENCE & EVALUATION**

Function to generate predictions from the model.

In [57]:
def generate_prediction(model, tokenizer, input_text: str, config: EvalConfig) -> str:
    # Prepare the input using the chat template
    sample = [{"role": "user", "content": input_text}]
    prompt = tokenizer.apply_chat_template(
        conversation=sample, tokenize=False, add_generation_prompt=True
    )
    tokenized_input = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(config.device)

    # Generate output
    model.eval()
    with torch.no_grad():
        gen_output = model.generate(
            **tokenized_input,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=config.max_new_tokens,
            do_sample=config.do_sample,
            temperature=config.temperature,
            num_beams=config.num_beams,
        )
    
    input_len = tokenized_input["input_ids"].shape[1]
    new_tokens = gen_output[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

Define a map of programming languages to their corresponding file extensions.

In [ ]:
from collections import defaultdict

LANG_MAP = {
    "python": "python", "py": "python",
    "java": "java",
    "javascript": "javascript", "js": "javascript",
    "typescript": "typescript", "ts": "typescript",
    "go": "go", "golang": "go",
    "php": "php",
    "ruby": "ruby", "rb": "ruby",
    "c#": "c_sharp", "csharp": "c_sharp",
    "c": "c", "cpp": "c", "c++": "c",
}

Extract the language from the input prompt and determine the file extension.

In [ ]:
def extract_lang(input_text: str, default: str = "python") -> str:
    """Extract language from 'complete <lang>: ...' prefix."""
    first_token = input_text.split(":")[0].strip().lower()
    if first_token.startswith("complete "):
        lang_key = first_token[len("complete "):].strip()
        return LANG_MAP.get(lang_key, default)
    return default

Compute the CodeBLEU score for the generated code against the reference code.

In [ ]:
def _safe_codebleu(references, predictions, lang, weights, fallback="python") -> tuple[dict, str]:

    try:
        return calc_codebleu(references=references, predictions=predictions,
                             lang=lang, weights=weights), lang
    except (ImportError, TypeError) as e:
        print(f"    [warn] parser error for '{lang}' ({type(e).__name__}) — falling back to '{fallback}'")
        return calc_codebleu(references=references, predictions=predictions,
                             lang=fallback, weights=weights), fallback

Function that runs all the evaluation's pipeline and returns the CodeBLEU scores for the model.

In [ ]:
def run_evaluation(model, tokenizer, samples: list[dict], config: EvalConfig, model_name: str) -> dict:
    predictions, references, langs = [], [], []
    print(f"\nEvaluating {model_name} ({len(samples)} samples)...")

    for i, sample in enumerate(samples):
        pred = generate_prediction(model, tokenizer, sample["input"], config)
        predictions.append(pred)
        references.append(sample["reference"])
        langs.append(extract_lang(sample["input"], config.lang))
        if (i + 1) % 10 == 0 or (i + 1) == len(samples):
            print(f"  {i + 1}/{len(samples)} done")

    groups: dict = defaultdict(lambda: {"preds": [], "refs": []})
    for pred, ref, lang in zip(predictions, references, langs):
        groups[lang]["preds"].append(pred)
        groups[lang]["refs"].append(ref)

    total = len(predictions)
    agg = {k: 0.0 for k in ("codebleu", "ngram_match_score",
                              "weighted_ngram_match_score",
                              "syntax_match_score", "dataflow_match_score")}
    lang_scores: dict = {}
    for lang, data in groups.items():
        n = len(data["preds"])
        s, used_lang = _safe_codebleu(
            references=[[r] for r in data["refs"]],
            predictions=data["preds"],
            lang=lang,
            weights=config.weights,
        )
        lang_scores[lang] = {"scores": s, "n": n, "used_lang": used_lang}
        for k in agg:
            agg[k] += s[k] * n / total

    return {
        "model_name": model_name,
        "predictions": predictions,
        "references": references,
        "scores": agg,
        "lang_scores": lang_scores,
    }

Start the EvalConfig object and load the samples of the test dataset.

In [59]:
config = EvalConfig()
samples = load_test_samples(config)

Loading dataset: Juanxxo/smallcoder-dataset (split='test')
Using 100 of 100 samples


### **BASE MODEL EVALUATION**

In [60]:
base_results = run_evaluation(base_model, tokenizer, samples, config, "Qwen2.5-1.5B-Instruct (base)")


Evaluating Qwen2.5-1.5B-Instruct (base) (100 samples)...
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
    [warn] parser error for 'ruby' (TypeError) — falling back to 'python'
    [warn] parser error for 'java' (TypeError) — falling back to 'python'
    [warn] parser error for 'javascript' (TypeError) — falling back to 'python'
    [warn] parser error for 'go' (TypeError) — falling back to 'python'
    [warn] parser error for 'php' (TypeError) — falling back to 'python'


### **FINE-TUNED MODEL EVALUATION**

In [61]:
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL)
finetuned_model.eval()
print("Adapter loaded successfully.")

finetuned_results = run_evaluation(finetuned_model, tokenizer, samples, config, "qwen2.5-1.5B-code-adapter (fine-tuned)")

Adapter loaded successfully.

Evaluating qwen2.5-1.5B-code-adapter (fine-tuned) (100 samples)...
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
    [warn] parser error for 'ruby' (TypeError) — falling back to 'python'
    [warn] parser error for 'java' (TypeError) — falling back to 'python'
    [warn] parser error for 'javascript' (TypeError) — falling back to 'python'
    [warn] parser error for 'go' (TypeError) — falling back to 'python'
    [warn] parser error for 'php' (TypeError) — falling back to 'python'


## **RESULTS OF THE EVALUATION**

In [ ]:
def print_scores(result: dict):
    model_scores = result["scores"]
    print(f"\n  {result['model_name']}")
    print(f"    CodeBLEU:           {model_scores['codebleu']:.4f}")
    print(f"    n-gram match:       {model_scores['ngram_match_score']:.4f}")
    print(f"    weighted n-gram:    {model_scores['weighted_ngram_match_score']:.4f}")
    print(f"    syntax match:       {model_scores['syntax_match_score']:.4f}")
    print(f"    dataflow match:     {model_scores['dataflow_match_score']:.4f}")
    if result.get("lang_scores"):
        print("Per-language (weighted avg):")
        for lang, data in sorted(result["lang_scores"].items()):
            print(f"      {lang:12s} ({data['n']:3d} samples)  CodeBLEU={data['scores']['codebleu']:.4f}")

In [63]:
print("=" * 60)
print("CodeBLEU Benchmark — Qwen2.5-1.5B")
print("=" * 60)
print_scores(base_results)
print_scores(finetuned_results)

delta = finetuned_results["scores"]["codebleu"] - base_results["scores"]["codebleu"]
print(f"\n{'=' * 60}")
print(f"  Delta (fine-tuned − base):  {delta:+.4f}")
print(f"  Winner: {'Fine-tuned ✓' if delta > 0 else 'Base model'}")

CodeBLEU Benchmark — Qwen2.5-1.5B

  Qwen2.5-1.5B-Instruct (base)
    CodeBLEU:           0.2129
    n-gram match:       0.0163
    weighted n-gram:    0.0521
    syntax match:       0.3526
    dataflow match:     0.4307
    Per-language (weighted avg):
      go           (  9 samples)  CodeBLEU=0.3135
      java         ( 18 samples)  CodeBLEU=0.1703
      javascript   ( 13 samples)  CodeBLEU=0.2329
      php          (  6 samples)  CodeBLEU=0.3029
      python       ( 38 samples)  CodeBLEU=0.2046
      ruby         ( 16 samples)  CodeBLEU=0.1740

  qwen2.5-1.5B-code-adapter (fine-tuned)
    CodeBLEU:           0.2253
    n-gram match:       0.0925
    weighted n-gram:    0.1127
    syntax match:       0.3944
    dataflow match:     0.3017
    Per-language (weighted avg):
      go           (  9 samples)  CodeBLEU=0.3074
      java         ( 18 samples)  CodeBLEU=0.1662
      javascript   ( 13 samples)  CodeBLEU=0.2181
      php          (  6 samples)  CodeBLEU=0.2101
      python    

In [64]:
n_show = 1
print(f"\n{'=' * 60}")
print(f"Sample predictions (first {n_show} examples)")
print("=" * 60)

for i in range(min(n_show, len(samples))):
    print(f"\n--- Example {i + 1} ---")
    print(f"Input:\n{samples[i]['input'][:300]}")
    print(f"\nReference:\n{samples[i]['reference'][:300]}")
    print(f"\nBase model:\n{base_results['predictions'][i][:300]}")
    print(f"\nFine-tuned:\n{finetuned_results['predictions'][i][:300]}")


Sample predictions (first 1 examples)

--- Example 1 ---
Input:
complete ruby: def get_valid_skus_as_lazy(resource_group_name, resource_name, custom_headers:nil)

Reference:
      response = get_valid_skus_async(resource_group_name, resource_name, custom_headers:custom_headers).value!
      unless response.nil?
        page = response.body
        page.next_method = Proc.new do |next_page_link|
          get_valid_skus_next_async(next_page_link, custom_headers:custom_he

Base model:
To complete the Ruby method `get_valid_skus_as_lazy`, you need to define what it should do. Since no specific implementation is provided in your question, I'll create a simple example where we assume that `resource_group_name` and `resource_name` are passed as parameters and that the method returns 

Fine-tuned:
      get_valid_skus_async(resource_group_name, resource_name, custom_headers:custom_headers).lazy
    end
